# Inspeção manual da amostra (plano §3.2, item 4)

Casos ambíguos e suspeitos são resolvidos por **inspeção manual documentada**.
Este notebook lista os 100 selecionados, sinaliza suspeitos de não-software e
organiza os ambíguos para revisão.

**Fluxo de decisão** (auditável):
1. Inspecione os sinalizados abaixo (a coluna `url` abre o repositório).
2. Para excluir um repo: adicione-o a `config/sampling.yaml` →
   `exclusions.repos` com um comentário do motivo.
3. Re-execute `govscore sample` (rápido — buscas cacheadas) e depois
   `govscore run` (retomável — só os substitutos são extraídos).
4. `sensitivity`, `validate` e `make figures` regeneram o restante.

Como abrir: `make lab` (ou `uv run jupyter lab`) na raiz do repositório.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path.cwd()
while not (ROOT / "config" / "metrics.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_colwidth", 100)

d = yaml.safe_load((ROOT / "config" / "sample_full.yaml").read_text())
full = pd.DataFrame(d["full"])
amb = pd.DataFrame(d["ambiguous"])
full["url"] = "https://github.com/" + full.repo
print(f"selecionados: {len(full)} | ambíguos: {len(amb)} | gerado em {d['generated_at']}")
full.groupby("archetype").size()

selecionados: 100 | ambíguos: 406 | gerado em 2026-07-20


archetype
club          25
federation    25
stadium       25
toy           25
dtype: int64

## Suspeitos para inspeção

Duas heurísticas: padrão de nome típico de não-software (a triagem por
tópicos não pega tudo) e ausência de resposta humana nas issues amostradas
(sinal fraco de comunidade — vindo do QA da extração).

In [2]:
SUSPEITOS = (r"leetcode|interview|awesome|explore|roadmap|tutorial|study|"
             r"notes|book|course|cheat|free[-_]?vpn|subscription")
qa = pd.json_normalize(json.loads(
    (ROOT / "data/processed/full_metrics.json").read_text())["results"])
sem_resposta = set(qa.loc[qa["responsiveness.n_first_responses"] == 0, "repo"])

full["flag_nome"] = full.repo.str.lower().str.contains(SUSPEITOS, regex=True)
full["flag_sem_resposta"] = full.repo.isin(sem_resposta)
suspeitos = full[full.flag_nome | full.flag_sem_resposta]
suspeitos[["repo", "archetype", "language", "stars",
           "active_contributors_2plus", "flag_nome", "flag_sem_resposta", "url"]]

,repo,archetype,language,stars,active_contributors_2plus,flag_nome,flag_sem_resposta,url
0,torvalds/linux,federation,c,239888,101,False,True,https://github.com/torvalds/linux
12,FFmpeg/FFmpeg,federation,c,62223,100,False,True,https://github.com/FFmpeg/FFmpeg
16,git/git,federation,c,62105,100,False,True,https://github.com/git/git
36,MisterBooo/LeetCodeAnimation,stadium,java,76631,1,True,False,https://github.com/MisterBooo/LeetCodeAnimation
48,gitlabhq/gitlabhq,stadium,ruby,24501,2,False,True,https://github.com/gitlabhq/gitlabhq
55,github/explore,club,ruby,4819,72,True,True,https://github.com/github/explore
83,fustyles/Arduino,toy,c++,426,1,False,True,https://github.com/fustyles/Arduino
87,AITabby/opencodex,toy,typescript,377,3,False,True,https://github.com/AITabby/opencodex
93,yolfinance/yolfi-agent,toy,javascript,212,2,False,True,https://github.com/yolfinance/yolfi-agent
94,EFanZh/LeetCode,toy,rust,228,1,True,True,https://github.com/EFanZh/LeetCode


## Casos ambíguos registrados

Por motivo — os "fora das faixas" são as zonas deliberadas da matriz §3.1
(não são erro); truncados e inativos merecem olhada se algum substituto for
necessário.

In [3]:
amb.reason.str.slice(0, 40).value_counts()

reason
fora das faixas da §3.1                     354
contagem truncada em 3000 commits — conf     40
sem commits no branch default na janela      12
Name: count, dtype: int64

In [4]:
# ambíguos com contagem truncada (piso de atividade alto) — candidatos a
# inspeção caso um estrato precise de substituto
amb[amb.reason.str.startswith("contagem")].assign(
    url=lambda x: "https://github.com/" + x.repo
).sort_values("stars", ascending=False).head(20)

,repo,language,stars,active_contributors_2plus,classified_at,reason,url
6,openclaw/openclaw,typescript,383552,94,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/openclaw/openclaw
13,n8n-io/n8n,typescript,197153,88,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/n8n-io/n8n
18,microsoft/vscode,typescript,187717,73,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/microsoft/vscode
25,anomalyco/opencode,typescript,187713,32,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/anomalyco/opencode
79,open-webui/open-webui,python,146049,35,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/open-webui/open-webui
21,vercel/next.js,javascript,141024,48,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/vercel/next.js
24,denoland/deno,rust,107748,66,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/denoland/deno
34,oven-sh/bun,rust,94895,28,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/oven-sh/bun
30,bitcoin/bitcoin,c++,89654,71,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/bitcoin/bitcoin
15,spring-projects/spring-boot,java,81128,35,2026-07-20,contagem truncada em 3000 commits — confirmar manualmente,https://github.com/spring-projects/spring-boot


## Casos fora das heurísticas

A revisão adversarial encontrou 4 repositórios de conteúdo que nenhuma das
duas heurísticas captura. As categorias da inspeção (conteúdo, espelhos)
estão em `govscore.robustness` e alimentam os cenários de
`results/robustez.md` §4.1.

In [5]:
from govscore.robustness import CONTENT_REPOS, MIRROR_REPOS

categoria = {**{r: "conteúdo" for r in CONTENT_REPOS},
             **{r: "espelho" for r in MIRROR_REPOS}}
inspecionados = full[full.repo.isin(set(categoria) | set(suspeitos.repo))].copy()
inspecionados["categoria"] = inspecionados.repo.map(categoria).fillna(
    "software, sinal fraco")
inspecionados["fora_das_heuristicas"] = ~inspecionados.repo.isin(suspeitos.repo)
inspecionados.sort_values(["categoria", "archetype"])[
    ["repo", "archetype", "categoria", "fora_das_heuristicas", "stars", "url"]]

,repo,archetype,categoria,fora_das_heuristicas,stars,url
53,SwiftOldDriver/iOS-Weekly,club,conteúdo,True,4991,https://github.com/SwiftOldDriver/iOS-Weekly
55,github/explore,club,conteúdo,False,4819,https://github.com/github/explore
25,krahets/hello-algo,stadium,conteúdo,True,128604,https://github.com/krahets/hello-algo
26,danielmiessler/SecLists,stadium,conteúdo,True,72297,https://github.com/danielmiessler/SecLists
34,doocs/advanced-java,stadium,conteúdo,True,79000,https://github.com/doocs/advanced-java
36,MisterBooo/LeetCodeAnimation,stadium,conteúdo,False,76631,https://github.com/MisterBooo/LeetCodeAnimation
94,EFanZh/LeetCode,toy,conteúdo,False,228,https://github.com/EFanZh/LeetCode
95,Au1rxx/free-vpn-subscriptions,toy,conteúdo,False,350,https://github.com/Au1rxx/free-vpn-subscriptions
0,torvalds/linux,federation,espelho,False,239888,https://github.com/torvalds/linux
12,FFmpeg/FFmpeg,federation,espelho,False,62223,https://github.com/FFmpeg/FFmpeg


## Registro de decisões

Inspeção manual concluída em 2026-09-12. **Decisão: todos os 100 mantidos**
(nenhuma entrada nova em `exclusions.repos`). Justificativa e consequências
para o texto em `docs/decisions/2026-09-12-inspecao-manual-amostra.md`.

| data | repo | categoria | decisão | motivo |
|---|---|---|---|---|
| 2026-09-12 | torvalds/linux | espelho | mantido | software; governança fora do GitHub é limite do proxy (declarado) |
| 2026-09-12 | FFmpeg/FFmpeg | espelho | mantido | idem |
| 2026-09-12 | git/git | espelho | mantido | idem; D3 = 0 por artefatos de PR observados (assimetria declarada) |
| 2026-09-12 | gitlabhq/gitlabhq | espelho | mantido | idem |
| 2026-09-12 | MisterBooo/LeetCodeAnimation | conteúdo | mantido | atende aos critérios de inclusão; exclusão post hoc evitada |
| 2026-09-12 | EFanZh/LeetCode | conteúdo | mantido | idem |
| 2026-09-12 | github/explore | conteúdo | mantido | idem |
| 2026-09-12 | krahets/hello-algo | conteúdo | mantido | idem (fora das heurísticas) |
| 2026-09-12 | doocs/advanced-java | conteúdo | mantido | idem (fora das heurísticas) |
| 2026-09-12 | danielmiessler/SecLists | conteúdo | mantido | idem (fora das heurísticas) |
| 2026-09-12 | SwiftOldDriver/iOS-Weekly | conteúdo | mantido | idem (fora das heurísticas) |
| 2026-09-12 | Au1rxx/free-vpn-subscriptions | conteúdo | mantido | idem |
| 2026-09-12 | fustyles/Arduino | software, sinal fraco | mantido | silêncio nas issues é o fenômeno medido (D3 informativo) |
| 2026-09-12 | AITabby/opencodex | software, sinal fraco | mantido | idem |
| 2026-09-12 | yolfinance/yolfi-agent | software, sinal fraco | mantido | idem; sinal de stars declarado (robustez.md §7) |

Resultados com e sem cada categoria: `results/robustez.md` §4.1 — globais
robustos; Estádio×forks e Federação×Scorecard **não** robustos.